# Chapter 5 RAG in Agent

# 1. RAG solve problems
- Large knowledge base, frequently updated, and requires source citations √
- When a question requires reading the full text before answering (e.g., “summarize the entire book”) ×

In [1]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore
from psycopg.rows import dict_row
from psycopg_pool import ConnectionPool

load_dotenv()

model = ChatOpenAI(
    model=os.getenv("LLM_MODEL_ID", "deepseek-flash"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL", "https://api.deepseek.com"),
    temperature=0,
    extra_body={"thinking": {"type": "disabled"}},
)

# bge-m3 served by local Ollama through its OpenAI-compatible API
embeddings = OpenAIEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434/v1",
    api_key="ollama",                  # Ollama ignores the key, but the client needs one
    check_embedding_ctx_length=False,  # send raw text, not OpenAI token ids
)

# One connection pool shared by the checkpointer, the store and background workers
pool = ConnectionPool(
    os.getenv("DATABASE_URL"),
    max_size=10,
    kwargs={"autocommit": True, "prepare_threshold": 0, "row_factory": dict_row},
    open=True,
)

checkpointer = PostgresSaver(pool)          # short-term memory: thread state
store = PostgresStore(                      # long-term memory: per-user items
    pool,
    index={"dims": 1024, "embed": embeddings, "fields": ["content"]},
)
checkpointer.setup()   # creates tables on first run, no-op afterwards
store.setup()

## 1.1 Import

In [2]:
import hashlib
import re
import time

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

PDF_PATH = os.getenv("HELLO_AGENTS_PDF", os.path.join("..", "Hello-Agents-EN-V1.0.3-20260717.pdf"))   # put the book PDF in the repo root, or set HELLO_AGENTS_PDF
SOURCE = "hello-agents-en-v1.0.3"


def clean_title(title: str) -> str:
    return re.sub(r"(?<=[a-z])(?=[A-Z])", " ", title).strip()   # "AgentParadigms" -> "Agent Paradigms"


def load_pages(path: str) -> list[dict]:
    """One dict per page, tagged with the chapter and section it falls in (from the PDF outline)."""
    reader = PdfReader(path)
    marks = []

    def walk(items, depth):
        for it in items:
            if isinstance(it, list):
                walk(it, depth + 1)
            else:
                marks.append((reader.get_destination_page_number(it), depth, clean_title(it.title)))

    walk(reader.outline, 0)
    marks.sort(key=lambda m: m[0])

    pages, chapter, section, j = [], "", "", 0
    for i, page in enumerate(reader.pages):
        while j < len(marks) and marks[j][0] <= i:
            _, depth, title = marks[j]
            if depth == 0:
                chapter, section = title, ""
            elif depth == 1:
                section = title
            j += 1
        text = re.sub(r"[ \t]+", " ", page.extract_text() or "").strip()
        pages.append({"page": i + 1, "chapter": chapter, "section": section, "text": text})
    return pages


splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)


def make_chunks(pages: list[dict]) -> list[dict]:
    chunks = []
    for p in pages:
        for piece in splitter.split_text(p["text"]):
            if len(piece) < 50:          # page numbers, stray headers
                continue
            chunk_id = hashlib.sha1(f"{SOURCE}:{p['page']}:{piece}".encode()).hexdigest()
            chunks.append({"id": chunk_id, "source": SOURCE, "chapter": p["chapter"],
                           "section": p["section"], "page": p["page"], "content": piece})
    return chunks


SCHEMA = """
CREATE TABLE IF NOT EXISTS doc_chunks (
    id        text PRIMARY KEY,
    source    text NOT NULL,
    chapter   text,
    chapter_no int GENERATED ALWAYS AS (substring(chapter from 'Chapter ([0-9]+)')::int) STORED,
    section   text,
    page      int,
    content   text NOT NULL,
    embedding vector(1024) NOT NULL,
    tsv       tsvector GENERATED ALWAYS AS (to_tsvector('english', content)) STORED
);
CREATE INDEX IF NOT EXISTS doc_chunks_embedding_idx ON doc_chunks USING hnsw (embedding vector_cosine_ops);
CREATE INDEX IF NOT EXISTS doc_chunks_tsv_idx ON doc_chunks USING gin (tsv);
"""


def ingest(chunks: list[dict], batch_size: int = 64) -> int:
    with pool.connection() as conn:
        for statement in SCHEMA.split(";"):      # one statement per call: the pool prepares every query
            if statement.strip():
                conn.execute(statement)
        existing = {r["id"] for r in conn.execute("SELECT id FROM doc_chunks WHERE source = %s", (SOURCE,))}
    todo = [c for c in chunks if c["id"] not in existing]     # idempotent: only embed new chunks
    t0 = time.perf_counter()
    for i in range(0, len(todo), batch_size):
        batch = todo[i:i + batch_size]
        vectors = embeddings.embed_documents([c["content"] for c in batch])
        with pool.connection() as conn:
            for c, v in zip(batch, vectors):
                conn.execute(
                    "INSERT INTO doc_chunks (id, source, chapter, section, page, content, embedding) "
                    "VALUES (%s, %s, %s, %s, %s, %s, %s::vector) ON CONFLICT (id) DO NOTHING",
                    (c["id"], c["source"], c["chapter"], c["section"], c["page"], c["content"], str(v)),
                )
        if (i // batch_size) % 5 == 0:
            print(f"  embedded {i + len(batch)}/{len(todo)} ({time.perf_counter() - t0:.0f}s)")
    return len(todo)

- Chapter metadata comes from the PDF outline. Use reader.outline to tag each page with its chapter and section. This allows search results to be filtered by chapter and citations to be labeled as "Chapter X, page Y"
- Chunk by page. This makes page citations simplest, at the cost of splitting paragraphs that span pages; a 150-character overlap between chunks mitigates this somewhat.
- A single table supports both retrieval modes.
  - The embedding column has an HNSW index for vector search.
  - tsv is an auto-generated full-text search column with a GIN index.
  - chapter_no is also auto-generated, extracting the chapter number from the chapter title
- Ingestion is idempotent. A chunk’s id is a hash of the source, page number, and content concatenated together, so rerunning the import only embeds newly added chunks


In [3]:
t0 = time.perf_counter()
pages = load_pages(PDF_PATH)
print(f"pages: {len(pages)} ({time.perf_counter() - t0:.0f}s)")
print("page 320:", pages[319]["chapter"], "|", pages[319]["section"])
chunks = make_chunks(pages)
print(f"chunks: {len(chunks)}, avg {sum(len(c['content']) for c in chunks) // len(chunks)} chars")
print("new chunks embedded:", ingest(chunks))
with pool.connection() as conn:
    print(conn.execute("SELECT count(*) AS chunks, count(DISTINCT chapter) AS chapters FROM doc_chunks").fetchone())

pages: 806 (124s)
page 320: Chapter 8 Memory and Retrieval | 8.2 Memory System: Giving Agents Memory
chunks: 1834, avg 787 chars
new chunks embedded: 0
{'chunks': 1834, 'chapters': 18}


## 1.2 Retrieval: vectors, keywords, BM25, hybrid

In [4]:
from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi

COLUMNS = "id, chapter_no, section, page, content"


def vector_search(query: str, k: int = 10, chapter_no: int | None = None) -> list[dict]:
    """Nearest chunks by cosine distance (pgvector HNSW index)."""
    qv = str(embeddings.embed_query(query))
    where = "WHERE chapter_no = %(ch)s" if chapter_no else ""
    sql = (f"SELECT {COLUMNS}, 1 - (embedding <=> %(qv)s::vector) AS score FROM doc_chunks {where} "
           "ORDER BY embedding <=> %(qv)s::vector LIMIT %(k)s")
    with pool.connection() as conn:
        return conn.execute(sql, {"qv": qv, "k": k, "ch": chapter_no}).fetchall()


def keyword_search(query: str, k: int = 10, chapter_no: int | None = None) -> list[dict]:
    """Full-text search: any query term may match (OR), ranked by ts_rank_cd."""
    where = "AND chapter_no = %(ch)s" if chapter_no else ""
    sql = (f"SELECT {COLUMNS}, ts_rank_cd(tsv, query.q) AS score FROM doc_chunks, "
           "(SELECT replace(plainto_tsquery('english', %(q)s)::text, '&', '|')::tsquery AS q) AS query "
           f"WHERE tsv @@ query.q {where} ORDER BY score DESC LIMIT %(k)s")
    with pool.connection() as conn:
        return conn.execute(sql, {"q": query, "k": k, "ch": chapter_no}).fetchall()


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9_]+", text.lower())   # keeps identifiers like chunk_overlap


with pool.connection() as conn:   # 1834 chunks fit in memory; rebuild the index when documents change
    corpus = conn.execute(f"SELECT {COLUMNS} FROM doc_chunks ORDER BY id").fetchall()
bm25 = BM25Okapi([tokenize(d["content"]) for d in corpus])


def bm25_search(query: str, k: int = 10, chapter_no: int | None = None) -> list[dict]:
    """Real BM25: rare terms weigh more (IDF), long chunks are normalized."""
    scores = bm25.get_scores(tokenize(query))
    results = []
    for i in scores.argsort()[::-1]:
        if scores[i] <= 0 or len(results) == k:
            break
        if chapter_no is None or corpus[i]["chapter_no"] == chapter_no:
            results.append(corpus[i] | {"score": float(scores[i])})
    return results


def rrf(result_lists: list[list[dict]], k: int = 5, c: int = 60) -> list[dict]:
    """Reciprocal Rank Fusion: score = sum of 1 / (c + rank) over all lists."""
    scores, docs = {}, {}
    for results in result_lists:
        for rank, d in enumerate(results, 1):
            scores[d["id"]] = scores.get(d["id"], 0) + 1 / (c + rank)
            docs[d["id"]] = d
    best = sorted(scores, key=scores.get, reverse=True)[:k]
    return [docs[i] | {"score": scores[i]} for i in best]


def hybrid_search(query: str, k: int = 5, chapter_no: int | None = None, pool_k: int = 20) -> list[dict]:
    return rrf([vector_search(query, pool_k, chapter_no), bm25_search(query, pool_k, chapter_no)], k)


def show(results: list[dict], width: int = 90):
    for d in results:
        text = " ".join(d["content"].split())[:width]
        print(f"  {d['score']:.3f}  p.{d['page']:<4} {text}")

## 1.4 The comparison of 4 retrieval methods

In [5]:
for q in ["Which AutoGen condition stops the chat when the word TERMINATE appears?",
          "为什么上下文越长，模型的表现反而会下降？"]:
    print("Q:", q)
    for name, search in [("vector", vector_search), ("pg keyword", keyword_search),
                         ("bm25", bm25_search), ("hybrid", hybrid_search)]:
        print(f" {name}:")
        show(search(q, 3), 75)

Q: Which AutoGen condition stops the chat when the word TERMINATE appears?
 vector:
  0.641  p.219  process ends. Here we set that when any message contains the keyword "TERMI
  0.545  p.219  (3) Define Team Collaboration Process In this case, the software developmen
  0.539  p.214  4. The group chat adds the new reply to the conversation history and activa
 pg keyword:
  1.400  p.65   that the next word is after word appears. According to maximum likelihood e
  1.200  p.66   Step 2: Calculate conditional probability This is the number of times the w
  1.100  p.65   built on this assumption are called N-gram models. Here, "N" represents the
 bm25:
  29.526  p.219  (3) Define Team Collaboration Process In this case, the software developmen
  23.502  p.65   that the next word is after word appears. According to maximum likelihood e
  22.333  p.66   Step 2: Calculate conditional probability This is the number of times the w
 hybrid:
  0.033  p.219  (3) Define Team Collaboration Process In

Three things:
- Postgres full-text search is not BM25. ts_rank_cd only looks ate term frequency and position; it has no IDF, meaning it does not give rarer terms higher weight. As a result, common words like “word” and “appears” pulled the result toward the paragraph about N-grams on page 65. A true BM25 would give more weight to rare terms like “AutoGen” and “TERMINATE,” and the top result would be the correct page 219. If you say in an interview, “I implemented BM25 using Postgres full-text search,” that is a mistake.
- When a Chinese question queries an English book, keyword search finds nothing. The reason is that the terms simply do not match. bge-m3, however, is a multilingual model, so vector retrieval can match across languages and still found page 368, which discusses context rot.
- RRF only looks at rankings, not raw scores. Vector cosine similarity (0 to 1) and BM25 scores (which might be 29.5) are not on the same scale and cannot be added directly. So RRF scores only by each result’s rank in its respective list: 1 / (60 + rank), and then sums the scores across lists.

## 1.5 Query rewriting and reranking

In [6]:
class Queries(BaseModel):
    queries: list[str] = Field(description="3 short English search queries, each using different keywords")


query_writer = model.with_structured_output(Queries, method="function_calling")


def multi_query_search(question: str, k: int = 5, chapter_no: int | None = None) -> list[dict]:
    """Rewrite the question into several English queries, search each, fuse with RRF."""
    queries = query_writer.invoke(
        "Write 3 short English search queries for finding the answer to this question "
        f"in an English tutorial about building AI agents.\nQuestion: {question}").queries
    lists = []
    for q in [question, *queries]:
        lists += [vector_search(q, 20, chapter_no), bm25_search(q, 20, chapter_no)]
    return rrf(lists, k)


class Ranking(BaseModel):
    ids: list[int] = Field(description="Numbers of the passages that help answer the question, most useful first")


ranker = model.with_structured_output(Ranking, method="function_calling")


def llm_rerank(question: str, docs: list[dict], top_n: int = 4) -> list[dict]:
    """Listwise rerank with the LLM: it reads every candidate and orders the useful ones."""
    listing = "\n\n".join(f"[{i}] {d['content'][:600]}" for i, d in enumerate(docs))
    ids = ranker.invoke(f"Question: {question}\n\nPassages:\n{listing}\n\n"
                        "Return the numbers of the passages that help answer the question, most useful first.").ids
    return [docs[i] for i in ids if 0 <= i < len(docs)][:top_n]

- Multi-query rewriting: First have the LLM rewrite the question into three English search queries, run retrieval for each one, and then fuse the results with RRF. After this step, keyword retrieval can also be used for Chinese questions.

- Reranking: The standard approach in production is to use a cross-encoder model, such as bge-reranker-v2-m3, Cohere Rerank, or Jina Rerank. However, Ollama does not provide a rerank API, so here the LLM reads all candidates at once and then orders them; this is called listwise reranking. The cost is one additional LLM call.

## 1.6 validate retrieval performance

In [7]:
EVAL_SET = [
    # exact terms: keyword search should shine
    {"q": "What does GSSC stand for in the ContextBuilder?", "evidence": "gather-select-structure-compress"},
    {"q": "What does BFCL stand for?", "evidence": "berkeley function calling leaderboard"},
    {"q": "Which AutoGen condition stops the chat when the word TERMINATE appears?", "evidence": "textmentiontermination"},
    {"q": "What chunk_size and chunk_overlap does the RAG pipeline use?", "evidence": "chunk_overlap=200"},
    # paraphrases: no shared keywords, vector search should shine
    {"q": "Why does stuffing more and more tokens into the context window make the model worse?", "evidence": "context rot"},
    {"q": "What is the trick of writing a fake answer first and searching with its embedding?", "evidence": "hypothetical document embeddings"},
    {"q": "Which thinker argued that intelligence emerges from many simple mindless agents working together?", "evidence": "society of mind"},
    {"q": "How does CAMEL keep two role-playing agents on task without a human in the loop?", "evidence": "inception prompting"},
    {"q": "Which RL algorithm drops the value model and uses group-relative rewards instead?", "evidence": "no need for value model"},
    # Chinese questions over an English book: keyword search can't match at all
    {"q": "为什么上下文越长，模型的表现反而会下降？", "evidence": "context rot"},
    {"q": "多查询扩展是怎么提高检索召回率的？", "evidence": "multi-query expansion (mqe) is a technique"},
    {"q": "第 12 章用什么指标比较生成的题目和真实题目谁更好？", "evidence": "win rate"},
    {"q": "深度研究智能体用的是什么研究范式？", "evidence": "todo-driven research"},
    {"q": "GRPO 相比 PPO 去掉了哪个模型？", "evidence": "no need for value model"},
]


def normalize(text: str) -> str:
    return " ".join(text.lower().split())


def first_hit(results: list[dict], evidence: str) -> int | None:
    """1-based rank of the first chunk that contains the evidence string."""
    for rank, d in enumerate(results, 1):
        if evidence in normalize(d["content"]):
            return rank
    return None


METHODS = {
    "vector": lambda q: vector_search(q, 5),
    "pg keyword": lambda q: keyword_search(q, 5),
    "bm25": lambda q: bm25_search(q, 5),
    "hybrid": lambda q: hybrid_search(q, 5),
    "multi-query": lambda q: multi_query_search(q, 5),
    "hybrid+rerank": lambda q: llm_rerank(q, hybrid_search(q, k=12), top_n=5),
}


def eval_retrieval(methods: dict = METHODS, eval_set: list[dict] = EVAL_SET):
    rows = {}
    for name, search in methods.items():
        t0 = time.perf_counter()
        ranks = [first_hit(search(item["q"]), item["evidence"]) for item in eval_set]
        hit = sum(r is not None for r in ranks) / len(ranks)
        mrr = sum(1 / r for r in ranks if r) / len(ranks)
        rows[name] = ranks
        print(f"{name:<14} Hit@5 {hit:.2f}  MRR {mrr:.2f}  ({(time.perf_counter() - t0) / len(eval_set):.1f}s/query)  "
              + " ".join(str(r) if r else "-" for r in ranks))
    return rows

In [8]:
eval_retrieval()

vector         Hit@5 0.86  MRR 0.70  (0.1s/query)  1 - 2 2 1 1 2 1 1 3 1 1 - 1
pg keyword     Hit@5 0.57  MRR 0.36  (0.0s/query)  1 3 4 4 - - 5 1 1 - - - - 1
bm25           Hit@5 0.64  MRR 0.56  (0.0s/query)  1 - 1 1 1 - 3 1 1 - - 2 - 1
hybrid         Hit@5 0.93  MRR 0.71  (0.1s/query)  1 2 1 1 1 2 3 1 1 3 1 3 - 1
multi-query    Hit@5 0.93  MRR 0.93  (1.6s/query)  1 1 1 1 1 1 1 1 1 1 1 1 - 1
hybrid+rerank  Hit@5 1.00  MRR 0.86  (1.2s/query)  1 5 1 1 1 1 2 1 1 1 1 1 3 1


{'vector': [1, None, 2, 2, 1, 1, 2, 1, 1, 3, 1, 1, None, 1],
 'pg keyword': [1, 3, 4, 4, None, None, 5, 1, 1, None, None, None, None, 1],
 'bm25': [1, None, 1, 1, 1, None, 3, 1, 1, None, None, 2, None, 1],
 'hybrid': [1, 2, 1, 1, 1, 2, 3, 1, 1, 3, 1, 3, None, 1],
 'multi-query': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, None, 1],
 'hybrid+rerank': [1, 5, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 3, 1]}

**Hit@5:** Whether there is a hit among the top 5 results.

**MRR:** The reciprocal of the rank of the first hit, averaged over all questions. Rank 1 scores 1, rank 2 scores 0.5, and so on.

## 2.Generation and combination

### 2.1 Two-step RAG

In [9]:
ANSWER_PROMPT = """Answer the question using only the numbered passages from the Hello-Agents tutorial.
Cite the passages you use like [1] or [2][3].
If the passages don't contain the answer, say you couldn't find it in the tutorial. Don't guess.
Answer in the same language as the question."""


def format_passages(docs: list[dict]) -> str:
    return "\n\n".join(f"[{i}] (Ch.{d['chapter_no']}, p.{d['page']}) {d['content']}" for i, d in enumerate(docs, 1))


def generate(question: str, docs: list[dict]) -> str:
    return model.invoke([
        {"role": "system", "content": ANSWER_PROMPT},
        {"role": "user", "content": f"Passages:\n{format_passages(docs)}\n\nQuestion: {question}"},
    ]).content


def retrieve(question: str) -> list[dict]:
    return llm_rerank(question, hybrid_search(question, k=12), top_n=4)


def two_step_rag(question: str) -> str:
    docs = retrieve(question)
    print("  pages:", [d["page"] for d in docs])
    return generate(question, docs)

In [ ]:
for q in ["What does GSSC stand for, and what does each step do?", "为什么上下文越长，模型的表现反而会下降？"]:
    print("Q:", q)
    print("A:", two_step_rag(q), "\n")

The problem with the two-step approach surfaced in the first question. The explanations of the four GSSC steps were scattered across several chunks on pages 374–381, but retrieval only returned two passages, so the answer only clearly covered the first step. Fortunately, the prompt worked: the model honestly stated that the content of the remaining steps was not in the provided passages, and it did not fabricate anything.

This is the difference between being “grounded” and being “complete”: every sentence in the answer has a source, but the answer does not fully address the question.

### 2.2 Agentic RAG (Retrieval as tools)

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, wrap_model_call
from langchain.tools import tool


@tool
def search_docs(query: str, chapter: int | None = None) -> str:
    """Search the Hello-Agents tutorial, an English book about building AI agents.

    Args:
        query: A focused English search query. Translate non-English questions first.
        chapter: Optional chapter number (1-16) to search within.
    """
    docs = hybrid_search(query, k=5, chapter_no=chapter)
    return "\n\n".join(f"(Ch.{d['chapter_no']}, p.{d['page']}) {d['content']}" for d in docs) or "No passages found."


RAG_AGENT_PROMPT = """You are a tutor for the Hello-Agents tutorial, and you answer only from the tutorial.
For every question that isn't small talk, call search_docs first, even if you think you already know the answer.
Search more than once when a question has several parts, for example once per chapter when comparing.
Cite pages like (Ch.9, p.372). If the results don't cover the question, say the tutorial doesn't cover it.
Reply to small talk directly, without searching. Answer in the user's language."""

rag_agent = create_agent(model=model, tools=[search_docs], system_prompt=RAG_AGENT_PROMPT)


@wrap_model_call
def search_first(request: ModelRequest, handler):
    """Force a tool call on the first model call of each turn (the last message is the user's)."""
    if request.messages[-1].type == "human":
        request = request.override(tool_choice="any")
    return handler(request)


strict_agent = create_agent(model=model, tools=[search_docs], system_prompt=RAG_AGENT_PROMPT, middleware=[search_first])


def ask_agent(question: str, agent=rag_agent):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, {"recursion_limit": 12})
    for m in result["messages"]:
        for call in getattr(m, "tool_calls", None) or []:
            print(f"  tool call -> {call['name']}({call['args']})")
    print("REPLY:", result["messages"][-1].content)

In [11]:
ask_agent("比较一下第 8 章的 RAG 和第 9 章的上下文工程，它们分别解决什么问题？")
ask_agent("What is the capital of Australia?")
ask_agent("Hi! Thanks for helping me study yesterday.", strict_agent)
ask_agent("What is the capital of Australia?", strict_agent)

  tool call -> search_docs({'query': 'RAG retrieval augmented generation what problem it solves', 'chapter': 8})
  tool call -> search_docs({'query': 'context engineering what problem it solves', 'chapter': 9})
  tool call -> search_docs({'chapter': 9, 'query': 'context rot limited resource diminishing returns context window management strategies'})
  tool call -> search_docs({'chapter': 8, 'query': 'RAG limitations hallucination knowledge base external knowledge retrieval pipeline'})
REPLY: # 第 8 章 RAG vs. 第 9 章上下文工程

## 一、各自解决什么问题

**第 8 章 RAG：解决"模型不知道"的问题——知识缺口**

RAG 的核心思路是：在生成答案之前，先从外部知识库检索相关信息，再把检索结果作为上下文提供给大模型，从而生成更准确、更可靠的答案。它拆成三个词理解：Retrieval（从知识库查询相关内容）、Augmented（把检索结果整合进提示词）、Generation（输出兼顾准确性与透明性的答案）(Ch.8, p.337)。

它针对的是大模型的三类固有局限：知识时效性、领域知识缺失、事实准确性（通过检索验证减少幻觉），以及可解释性（提供信息来源）(Ch.8, p.310)。

工程上，RAG 是"五层七步"的流水线：用户层（RAGTool 统一接口）→ 应用层（智能问答、搜索、管理）→ 处理层（文档解析、分块、向量化）→ 存储层（向量库、文档存储）→ 基础层（Embedding 模型、LLM、数据库）(Ch.8, p.341)。完整流程是：任意格式文档 → MarkItDown 转换 → Markdown 文本 → 智能分块 → 向量化 → 存

- The strength of Agentic RAG is its flexibility. For comparison questions, it decided on its own to retrieve four times, filtering by chapter each time, and finally assembled a comparison table. A fixed pipeline cannot do this.
- Its risk is that it may skip retrieval. The prompt explicitly stated, “Even if you think you know the answer, retrieve first,” yet the model still answered “Canberra” directly and even gave itself a justification: “It’s common knowledge.” 
- Rules that must be followed need to be enforced in code. The search_first middleware sets tool_choice="any" on the first model call of each turn, forcing the model to call a tool, so out-of-scope (OOS) questions were correctly refused. The cost is that even casual chit-chat triggers an unnecessary retrieval (“study yesterday”).

### 2.3 Corrective RAG (It will inspect its own retrieval graph)

In [12]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class RAGState(TypedDict, total=False):
    question: str
    queries: list[str]
    docs: list[dict]
    attempts: int
    enough: bool
    missing: str
    answer: str
    grounded: bool


class Grade(BaseModel):
    enough: bool = Field(description="True only if the passages contain everything needed to fully answer the question")
    missing: str = Field(description="What information is still missing, or 'nothing'")


class Rewrite(BaseModel):
    queries: list[str] = Field(description="1-3 short English search queries, one per missing piece")


class Grounding(BaseModel):
    grounded: bool = Field(description="True if every claim in the answer is supported by the passages")


grader = model.with_structured_output(Grade, method="function_calling")
rewriter = model.with_structured_output(Rewrite, method="function_calling")
grounding_judge = model.with_structured_output(Grounding, method="function_calling")

MAX_ATTEMPTS = 2


def retrieve_node(state: RAGState) -> RAGState:
    per_query = 4 if len(state["queries"]) == 1 else 2
    found = []
    for q in state["queries"]:                    # each sub-query is searched and reranked on its own
        found += llm_rerank(q, hybrid_search(q, k=10), top_n=per_query)
    seen = {d["id"] for d in state.get("docs", [])}
    docs = state.get("docs", []) + [d for d in found if d["id"] not in seen]   # keep earlier finds
    attempts = state.get("attempts", 0) + 1
    print(f"  retrieve #{attempts}: {state['queries']} -> new pages {[d['page'] for d in found if d['id'] not in seen]}")
    return {"docs": docs[:10], "attempts": attempts}             # cap the context, keep the earliest finds


def grade_node(state: RAGState) -> RAGState:
    g = grader.invoke(f"Question: {state['question']}\n\nPassages:\n{format_passages(state['docs'])}")
    print(f"  grade: enough={g.enough}, missing={g.missing!r}")
    return {"enough": g.enough, "missing": g.missing}


def rewrite_node(state: RAGState) -> RAGState:
    r = rewriter.invoke(f"Question: {state['question']}\nAlready searched: {state['queries']}\n"
                        f"Still missing: {state['missing']}\n"
                        "Write one short English search query per missing piece. Use only specific terms "
                        "from the topic; don't add generic words like 'tutorial', 'AI', 'agent' or 'explanation'.")
    queries = r.queries[:3]                       # the schema says 1-3, but nothing enforces it
    print(f"  rewrite -> {queries}")
    return {"queries": queries}


def generate_node(state: RAGState) -> RAGState:
    question = state["question"]
    if not state.get("enough"):
        question += "\n(The passages may not cover everything. Answer only what they support and say what is missing.)"
    return {"answer": generate(question, state["docs"])}


def check_node(state: RAGState) -> RAGState:
    v = grounding_judge.invoke(f"Passages:\n{format_passages(state['docs'])}\n\nAnswer:\n{state['answer']}")
    print(f"  grounded: {v.grounded}")
    return {"grounded": v.grounded}


def after_grade(state: RAGState) -> str:
    if state["enough"] or state["attempts"] >= MAX_ATTEMPTS:
        return "generate"
    return "rewrite"


builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("grade", grade_node)
builder.add_node("rewrite", rewrite_node)
builder.add_node("generate", generate_node)
builder.add_node("check", check_node)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges("grade", after_grade, ["generate", "rewrite"])
builder.add_edge("rewrite", "retrieve")
builder.add_edge("generate", "check")
builder.add_edge("check", END)
corrective_rag = builder.compile()


def ask_graph(question: str) -> RAGState:
    state = corrective_rag.invoke({"question": question, "queries": [question]})
    print("ANSWER:", state["answer"])
    return state

Grade node put the answer into state, and then read by after_grade

In [13]:
ask_graph("What does GSSC stand for, and what does each step do?")
ask_graph("What is the capital of Australia?")

  retrieve #1: ['What does GSSC stand for, and what does each step do?'] -> new pages [432, 374, 97, 651]
  grade: enough=False, missing='The passages give the GSSC acronym (Gather-Select-Structure-Compress) and describe the Gather stage (multi-source information collection, fault tolerance/flexibility) and mention Select and Compress stages only in passing. They do not explain what the Select, Structure, and Compress stages each do in detail.'
  rewrite -> ['GSSC Select stage information filtering relevance', 'GSSC Structure stage organizing selected information', 'GSSC Compress stage context compression summarization']
  retrieve #2: ['GSSC Select stage information filtering relevance', 'GSSC Structure stage organizing selected information', 'GSSC Compress stage context compression summarization'] -> new pages [379, 378, 380, 379, 380, 381]
  grade: enough=True, missing='nothing'
  grounded: True
ANSWER: GSSC stands for **Gather-Select-Structure-Compress**, a pipeline that decomposes

{'question': 'What is the capital of Australia?',
 'queries': ['capital of Australia Canberra'],
 'docs': [{'id': '7b1ef5e5bf9f6627fae5252f21049c777f2691e5',
   'chapter_no': 8,
   'section': '8.1 From Cognitive Science to Agent Memory',
   'page': 309,
   'content': 'Declarative Memory: Knowledge that can be expressed in language, further divided into:\nSemantic Memory: General knowledge and concepts (such as "Paris is the capital of\nFrance")\nEpisodic Memory: Personal experiences and events (such as "yesterday\'s meeting\ncontent")\n8.1.2 Why Agents Need Memory and RAG\nDrawing on the design of human memory systems, we can understand why agents also need similar\nmemory capabilities. An important characteristic of human intelligence is the ability to remember past\nexperiences, learn from them, and apply these experiences to new situations. Similarly, a truly\nintelligent agent also needs memory capabilities. For LLM-based agents, they typically face two\nfundamental limitations: fo

### 2.4 The agent with users' memory and knowlege base

In [14]:
import uuid
from dataclasses import dataclass
from datetime import date
from typing import Literal

from langchain.tools import ToolRuntime, tool


@dataclass
class UserContext:
    user_id: str   # filled by the server after auth, never by the model


@tool
def save_memory(content: str, kind: Literal["fact", "episode"], importance: float,
                runtime: ToolRuntime[UserContext]) -> str:
    """Save something worth remembering about the user for future conversations.

    Args:
        content: The full information to remember, in one self-contained sentence.
        kind: "fact" for lasting knowledge or preferences, "episode" for a dated event.
        importance: How important it is, from 0.0 to 1.0.
    """
    key = str(uuid.uuid4())
    runtime.store.put((runtime.context.user_id, "memories"), key,
                      {"content": content, "kind": kind, "importance": importance})
    return f"Saved {kind} {key[:8]}: {content}"


@tool
def search_memories(query: str, runtime: ToolRuntime[UserContext]) -> str:
    """Search what you remember about the user. Use it before saying you don't know something.

    Args:
        query: What to look for, e.g. "user's travel plans".
    """
    items = runtime.store.search((runtime.context.user_id, "memories"), query=query, limit=5)
    lines = [f"[{i.value['kind']}] {i.value['content']} (score {i.score:.2f})" for i in items]
    return "\n".join(lines) or "No memories found."


def print_turn(result):
    """Print this turn's tool calls and the final reply."""
    last_human = max(i for i, m in enumerate(result["messages"]) if m.type == "human")
    for m in result["messages"][last_human:]:
        for call in getattr(m, "tool_calls", None) or []:
            print(f"  tool call -> {call['name']}({call['args']})")
    print("REPLY:", result["messages"][-1].content)

In [15]:
from pydantic import BaseModel, Field


# ---------- profile: one document per user, updated with patches ----------
class UserProfile(BaseModel):
    """Every field is optional: in a patch, None means "unchanged"."""
    name: str | None = None
    job: str | None = Field(None, description="Current job title and employer")
    languages: list[str] | None = Field(None, description="Full list of programming languages the user likes")
    answer_style: str | None = Field(None, description="How the user wants answers written, e.g. 'short, code first'")


profile_writer = model.with_structured_output(UserProfile, method="function_calling")

PROFILE_PROMPT = """You maintain a user profile.
Current profile (JSON): {profile}

Read the conversation below and return a patch: fill only the fields the user changed or revealed, leave the rest null.
Only use what the user said, not the assistant."""


def update_profile(user_id: str, conversation: str) -> UserProfile:
    item = store.get((user_id, "profile"), "main")
    old = UserProfile(**item.value) if item else UserProfile()
    patch = profile_writer.invoke([
        {"role": "system", "content": PROFILE_PROMPT.format(profile=old.model_dump_json())},
        {"role": "user", "content": conversation},
    ])
    new = old.model_copy(update=patch.model_dump(exclude_none=True))   # the code merges, not the LLM
    if new != old:
        store.put((user_id, "profile"), "main", new.model_dump(), index=False)
        print(f"  profile patch: {patch.model_dump(exclude_none=True)}")
    return new


# ---------- collection: many small memories, kept clean with ADD/UPDATE/DELETE/NOOP ----------
class Facts(BaseModel):
    facts: list[str] = Field(description="Self-contained facts about the user. Empty list if none.")


class MemoryDecision(BaseModel):
    action: Literal["ADD", "UPDATE", "DELETE", "NOOP"]
    target_id: str | None = Field(None, description="Id of the existing memory to update or delete")
    content: str | None = Field(None, description="Final memory text, for ADD and UPDATE")
    kind: Literal["fact", "episode"] = Field("fact", description="'episode' for dated events and plans")
    importance: float = Field(0.5, ge=0, le=1)


fact_extractor = model.with_structured_output(Facts, method="function_calling")
decider = model.with_structured_output(MemoryDecision, method="function_calling")

EXTRACT_PROMPT = """Extract facts about the user worth remembering in future sessions: projects, plans, dated events, goals and preferences.
Do not extract anything the profile already covers (name, job, favourite programming languages, answer style).
Current profile: {profile}
Skip small talk and anything only the assistant said.
Today is {today}. Turn relative dates into absolute dates.
Write each fact as one self-contained sentence about "the user"."""

DECIDE_PROMPT = """You manage a user's long-term memory.
New fact: {fact}

Existing memories most similar to it:
{existing}

Choose one action:
- ADD: the fact is new information.
- UPDATE: it changes a detail of an existing memory, such as a new date. Give target_id and the merged content.
- DELETE: the new fact contradicts an existing memory or says it no longer holds, for example a plan was cancelled.
  Give target_id. Do not keep a record of the cancellation.
- NOOP: it is already covered, including restatements and confirmations."""


def update_memories(user_id: str, conversation: str, thread_id: str, profile: UserProfile) -> None:
    ns = (user_id, "memories")
    facts = fact_extractor.invoke([
        {"role": "system", "content": EXTRACT_PROMPT.format(today=date.today().isoformat(), profile=profile.model_dump_json())},
        {"role": "user", "content": conversation},
    ]).facts
    for fact in facts:
        similar = store.search(ns, query=fact, limit=3)
        existing = "\n".join(f"- id={i.key} | {i.value['content']}" for i in similar) or "(none)"
        d = decider.invoke(DECIDE_PROMPT.format(fact=fact, existing=existing))
        known = {i.key for i in similar}
        if d.action == "ADD":
            store.put(ns, str(uuid.uuid4()), {"content": d.content or fact, "kind": d.kind,
                                              "importance": d.importance, "source_thread": thread_id})
        elif d.action == "UPDATE" and d.target_id in known:
            store.put(ns, d.target_id, {"content": d.content, "kind": d.kind,
                                        "importance": d.importance, "source_thread": thread_id})
        elif d.action == "DELETE" and d.target_id in known:
            store.delete(ns, d.target_id)
        target = f" {d.target_id[:8]}" if d.target_id else ""
        print(f"  {d.action}{target}: {d.content or fact}")


def remember(user_id: str, conversation: str, thread_id: str) -> None:
    """Background job run after a turn: refresh the profile, then the memory collection."""
    profile = update_profile(user_id, conversation)
    update_memories(user_id, conversation, thread_id, profile)


def show_memory(user_id: str) -> None:
    item = store.get((user_id, "profile"), "main")
    print("PROFILE:", item.value if item else None)
    for i in store.search((user_id, "memories"), limit=50):
        print(f"  [{i.value['kind']}] {i.key[:8]} {i.value['content']}")

In [16]:
import json
from datetime import datetime, timezone

from langchain.agents.middleware import ModelRequest, dynamic_prompt


def rerank(items, half_life_days: float = 30, min_similarity: float = 0.4):
    """Score = 0.7 * similarity + 0.2 * recency + 0.1 * importance. Drops weak matches."""
    now = datetime.now(timezone.utc)
    scored = []
    for item in items:
        if item.score is None or item.score < min_similarity:
            continue
        age_days = (now - item.updated_at).total_seconds() / 86400
        recency = 0.5 ** (age_days / half_life_days)          # halves every 30 days
        importance = item.value.get("importance", 0.5)
        final = 0.7 * item.score + 0.2 * recency + 0.1 * importance
        scored.append((final, item.score, recency, item))
    return sorted(scored, key=lambda row: row[0], reverse=True)


BASE_PROMPT = ("You are a helpful assistant with long-term memory about the user. "
               "The profile and memories below were loaded for this message, so use them directly; "
               "call search_memories only when they don't answer the question. "
               "Use a memory only when it helps with the current message; never bring it up unprompted.")


@dynamic_prompt
def memory_prompt(request: ModelRequest) -> str:
    """Build the system prompt before every model call: profile + top memories."""
    user_id = request.runtime.context.user_id
    store = request.runtime.store
    parts = [BASE_PROMPT, f"Today is {date.today().isoformat()}."]

    item = store.get((user_id, "profile"), "main")
    profile = {k: v for k, v in (item.value if item else {}).items() if v}
    if profile:
        parts.append(f"User profile: {json.dumps(profile, ensure_ascii=False)}")
    if profile.get("answer_style"):
        parts.append(f"Always follow the user's preferred answer style: {profile['answer_style']}.")

    query = next(m.text for m in reversed(request.messages) if m.type == "human")
    top = rerank(store.search((user_id, "memories"), query=query, limit=10))[:3]
    if top:
        parts.append("Relevant memories:\n" + "\n".join(f"- {row[3].value['content']}" for row in top))
    return "\n\n".join(parts)

In [17]:
import json
from datetime import date

from langchain.agents.middleware import dynamic_prompt


@dynamic_prompt
def tutor_prompt(request: ModelRequest) -> str:
    """Tutor rules + what we know about this user. Documents are fetched by the search_docs tool."""
    user_id = request.runtime.context.user_id
    store = request.runtime.store
    parts = [RAG_AGENT_PROMPT, f"Today is {date.today().isoformat()}."]

    item = store.get((user_id, "profile"), "main")
    profile = {k: v for k, v in (item.value if item else {}).items() if v}
    if profile:
        parts.append(f"User profile: {json.dumps(profile, ensure_ascii=False)}")
    if profile.get("answer_style"):
        parts.append(f"Always follow the user's preferred answer style: {profile['answer_style']}.")

    query = next(m.text for m in reversed(request.messages) if m.type == "human")
    top = rerank(store.search((user_id, "memories"), query=query, limit=10))[:3]
    if top:
        parts.append("What you remember about the user (use only when it helps):\n"
                     + "\n".join(f"- {row[3].value['content']}" for row in top))
    return "\n\n".join(parts)


tutor = create_agent(
    model=model,
    tools=[search_docs, search_memories],
    middleware=[tutor_prompt, search_first],
    checkpointer=checkpointer,
    store=store,
    context_schema=UserContext,
)


def tutor_chat(text: str, thread_id: str, user_id: str = "user123"):
    result = tutor.invoke(
        {"messages": [{"role": "user", "content": text}]},
        {"configurable": {"thread_id": thread_id}, "metadata": {"user_id": user_id}, "recursion_limit": 12},
        context=UserContext(user_id=user_id),
    )
    print(f"USER: {text}")
    print_turn(result)

In [18]:
remember("user123", "user: I'm Zhang San, an agent engineer. I have agent engineer interviews next month. "
                    "Please keep answers short and show code first.", "seed-rag")
show_memory("user123")
tutor_chat("RAG 里的 HyDE 是什么？怎么实现？", "tutor-1")
tutor_chat("这个知识点面试时可能会怎么问？", "tutor-1")

  profile patch: {'name': 'Zhang San', 'job': 'Agent engineer', 'answer_style': 'short, code first'}
  ADD: The user has agent engineer interviews scheduled for November 2026.
PROFILE: {'job': 'Agent engineer', 'name': 'Zhang San', 'languages': None, 'answer_style': 'short, code first'}
  [episode] ecca4d35 The user has agent engineer interviews scheduled for November 2026.
USER: RAG 里的 HyDE 是什么？怎么实现？
  tool call -> search_docs({'query': 'HyDE hypothetical document embeddings RAG'})
  tool call -> search_docs({'query': 'query rewriting retrieval optimization implementation'})
REPLY: **HyDE = 用“假答案”去检索真文档**（Ch.8, p.350）

问题与文档在语义空间分布不同（问句 vs 陈述句），HyDE 让 LLM 先写一段假设答案，再用它的向量检索。

```python
def _prompt_hyde(query: str) -> Optional[str]:
    """生成假设文档以改进检索"""
    try:
        from ...core.llm import HelloAgentsLLM
        llm = HelloAgentsLLM()
        prompt = [
            {"role": "system", "content": "根据用户问题，先写一段可能的答案段落，"
                                          "用作向量检索的查询文档（不要分析过程）。"},

- The sources of two kinds of context play different roles:
  - knowledge base: retrieve by using `search_docs`, the answer cited reference
  - User memory: Injected by middleware before the start of each turn. The response adopted a “code-first” style; the mention of “interview” in the second question was directly linked to the memory “I have an interview next month,” so the model did not call search_memories; what “this concept” referred to came from the thread’s short-term memory.

- They also have an important differnce:
  - The permissions: memories are isolated by `user_id`; the knowledge base is public, and inproduction, must be filtered by document-level permissions
  - writers: Memories are written by the Agent after a conversation; the knowledge base is written only by the offline ingestion pipeline

## 3. Evaluate the answers

In [20]:
class Refusal(BaseModel):
    refused: bool = Field(
        description="True if the answer says the information was not found or is not covered"
    )


refusal_judge = model.with_structured_output(Refusal, method="function_calling")

OUT_OF_SCOPE = [
    "What is the capital of Australia?",
    "谁赢得了 2022 年世界杯？",
    "How do I bake sourdough bread?",
]


def judge_grounded(docs: list[dict], answer: str) -> bool:
    return grounding_judge.invoke(
        f"Passages:\n{format_passages(docs)}\n\nAnswer:\n{answer}"
    ).grounded


def run_two_step(question: str) -> dict:
    docs = retrieve(question)
    answer = generate(question, docs)
    return {
        "answer": answer,
        "grounded": judge_grounded(docs, answer),
    }


def run_corrective(question: str) -> dict:
    state = corrective_rag.invoke(
        {"question": question, "queries": [question]}
    )
    return {
        "answer": state["answer"],
        "grounded": state["grounded"],
        "attempts": state["attempts"],
    }


for name, run in [("two-step", run_two_step), ("corrective", run_corrective)]:
    grounded = [run(item["q"])["grounded"] for item in EVAL_SET]
    refused = [
        refusal_judge.invoke(f"Answer:\n{run(q)['answer']}").refused
        for q in OUT_OF_SCOPE
    ]
    print(
        f"{name}: grounded {sum(grounded)}/{len(grounded)}, "
        f"refused out-of-scope {sum(refused)}/{len(refused)}"
    )

two-step: grounded 14/14, refused out-of-scope 3/3
  retrieve #1: ['What does GSSC stand for in the ContextBuilder?'] -> new pages [366, 374, 432, 372]
  grade: enough=True, missing='nothing'
  grounded: True
  retrieve #1: ['What does BFCL stand for?'] -> new pages [577, 573, 578, 633]
  grade: enough=False, missing='The passages never spell out what the acronym "BFCL" stands for. Passage [1] shows a directory path "berkeley-function-call-leaderboard," which strongly implies "Berkeley Function Calling Leaderboard," but the expansion is not explicitly stated.'
  rewrite -> ['Berkeley Function Calling Leaderboard BFCL']
  retrieve #2: ['Berkeley Function Calling Leaderboard BFCL'] -> new pages [569, 572, 577]
  grade: enough=True, missing='nothing'
  grounded: True
  retrieve #1: ['Which AutoGen condition stops the chat when the word TERMINATE appears?'] -> new pages [219, 219, 221]
  grade: enough=True, missing='nothing'
  grounded: True
  retrieve #1: ['What chunk_size and chunk_overl

The two pipelines scored the same, but that does not mean they are equally good. A score of 14/14 only shows that nothing was fabricated; it cannot measure whether the answer was complete. For the GSSC question, the two-step pipeline covered only one step, yet it was still judged as grounded.

# Comparison of Three RAG Integration Approaches 

| | Two-step RAG | Agentic RAG | Corrective RAG |
|---|---|---|---|
| **Who decides retrieval** | Code: one fixed retrieval per turn | Model: decides whether to retrieve and how many times | Graph: after retrieval, judges whether the information is sufficient; if not, rewrites the query and retrieves again |
| **Minimum number of LLM calls** | 2 (reranking, generation) | 2+ (a call is needed at each step of the tool loop) | 4 (reranking, grading, generation, verification), increasing on retries |
| **Observed advantages** | Simple and stable; all out-of-scope questions are refused | Can decompose comparison questions (retrieve 4 times and filter by chapter) and write English retrieval queries on its own | Can complete multi-part questions (the GSSC question went from 1 step to 4 steps), with reliable refusal |
| **Observed disadvantages** | Incomplete answers to multi-part questions | When relying only on prompts, it may skip retrieval and answer from the model’s own knowledge | Slow and costly; grading and rewriting prompts require repeated tuning |
| **Suitable scenarios** | QA bots where every question requires document lookup | Mixed tasks and questions requiring multi-step reasoning | High-accuracy scenarios such as healthcare and regulations |

## What Is Still Missing for Production

- **BM25 is kept in memory, so it must be rebuilt whenever documents change.**  
  In production, OpenSearch or Elasticsearch is typically used. If you want to stay in Postgres, extensions such as ParadeDB’s `pg_search` and VectorChord-bm25 can be used.

- **LLM reranking is slow and expensive.**  
  It can be replaced with cross-encoder models such as `bge-reranker-v2-m3`, Cohere Rerank, or Jina Rerank.

- **Permission control.**  
  Add tenant and access-permission filters to SQL `WHERE` clauses, and exclude content the user is not authorized to see at the retrieval stage rather than filtering after generation.

- **Incremental updates.**  
  This lesson generates IDs from content hashes. When a document is updated, new chunks are created, but old chunks are not deleted. Three things still need to be handled:
  - Delete a document’s old chunks when it is updated.
  - Record the embedding model version.
  - Regenerate all vectors when changing the embedding model.

- **Chunking and context expansion.**
  - Chunking by heading structure is more reasonable than chunking by page.
  - When a chunk is retrieved, include its adjacent chunks as well (Assignment 3).
  - Use “retrieve small chunks, feed large chunks to the model” (parent-document retrieval).

- **PDF parsing quality.**  
  `pypdf` performs mediocrely on tables and code. Consider PyMuPDF, docling, unstructured, or MarkItDown.

- **Caching and latency budget.**  
  Cache query vectors and answers to common questions; limit the maximum number of retries for Corrective RAG.

- **Observability and regression testing.**  
  Use LangSmith or Langfuse to trace every retrieval. Continuously expand the evaluation set and put it in CI so it runs on every change.